In [ ]:
# !pip list

In [ ]:
# !pip install unsloth[colab-new] xformers trl peft accelerate bitsandbytes hf_transfer
# !pip install --upgrade transformers

In [33]:
!export HF_HUB_ENABLE_HF_TRANSFER=1

In [1]:
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template
import torch
from transformers import EarlyStoppingCallback

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Capability: {torch.cuda.get_device_capability(0)}")

PyTorch Version: 2.8.0+cu126
CUDA Available: True
CUDA Version: 12.6
cuDNN Version: 91002
GPU Name: Tesla T4
GPU Capability: (7, 5)


In [3]:

# ==============================================================================
# MODEL SETUP - Optimized for RTX 4090
# ==============================================================================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length=512*2,  # Optimal for RTX 4090 with your dataset
    dtype=None,  # Auto-detect optimal dtype (bfloat16 for RTX 4090)
    load_in_4bit=True,  # Enable 4-bit quantization for memory efficiency
    )



==((====))==  Unsloth 2025.9.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [4]:
print(tokenizer.pad_token)
print(tokenizer.padding_side)

<|finetune_right_pad_id|>
left


In [5]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,)

In [6]:
# ==============================================================================
# LORA CONFIGURATION - Optimized for 3B Model
# ==============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Rank - good balance for 3B model
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],  # All linear layers
    lora_alpha=32,  # Alpha = rank for balanced learning
    lora_dropout=0.05,  # Low dropout for stability
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=42,
    use_rslora=False,  # Standard LoRA works well for this case
    loftq_config=None,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.9.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [7]:

# ==============================================================================
# CHAT TEMPLATE SETUP - Llama 3.2 Compatible
# ==============================================================================

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.2",  # Llama 3.2
)



In [10]:
# ==============================================================================
# DATASET PREPARATION
# ==============================================================================

def format_rbi_dataset(examples):
    """
    Format RBI QA dataset with custom template (no automatic system headers)
    """
    texts = []

    for i in range(len(examples['question'])):
        system_msg = """You are a highly knowledgeable AI assistant with expertise in Indian banking and financial regulations,
                        particularly those outlined in Reserve Bank of India (RBI) circulars. Your task is to answer questions
                        based on the RBI circulars and related financial regulations."""

        # Manual template formatting to avoid automatic system headers
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

              {system_msg.strip()}<|eot_id|><|start_header_id|>user<|end_header_id|>

              {examples['question'][i]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

              {examples['answer'][i]}<|eot_id|>"""

        texts.append(text)

    return {"text": texts}


# Load your dataset
# Replace with your actual dataset loading method
dataset = load_dataset("Vishva007/RBI-Circular-QA-Dataset", split="train")

# Apply formatting
dataset = dataset.map(
    format_rbi_dataset,
    batched=True,
    remove_columns=dataset.column_names,  # Keep only 'text' column
)


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [11]:
print(dataset[5]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

              You are a highly knowledgeable AI assistant with expertise in Indian banking and financial regulations, 
                        particularly those outlined in Reserve Bank of India (RBI) circulars. Your task is to answer questions 
                        based on the RBI circulars and related financial regulations.<|eot_id|><|start_header_id|>user<|end_header_id|>

              What was the Reserve Bank of India's policy regarding the delayed submission of regulatory returns in light of the COVID-19 pandemic, and which types of returns were excluded from this policy?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

              Due to disruptions caused by the COVID-19 pandemic, the Reserve Bank of India allowed regulated entities to submit regulatory returns to the Department of Regulation with a delay of up to 30 days from the original due date. However, this extension did not apply to statutory ret

In [27]:
# ==============================================================================
# TRAINING CONFIGURATION - Optimized for RTX 4090
# ==============================================================================

training_args = TrainingArguments(
    # Output and logging
    output_dir="./rbi-llama-3.2-3b-finetuned",
    run_name="rbi-qa-finetune",

    # Training parameters - Optimized for RTX 4090 24GB
    per_device_train_batch_size=2,  # Small batch size for memory efficiency
    gradient_accumulation_steps=8,  # Effective batch size = 2 * 8 = 16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Learning parameters
    num_train_epochs=1,  # Good for QA task without overfitting
    max_steps=50,  # Use epochs instead
    learning_rate=1e-4,  # Standard for LoRA fine-tuning
    lr_scheduler_type="cosine",  # Smooth learning rate decay
    warmup_steps=100,  # steps for warmup

    # Optimization
    optim="paged_adamw_8bit",  # Memory efficient optimizer
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Memory optimization
    dataloader_pin_memory=False,  # Reduces VRAM usage
    fp16=True,  # Use bfloat16 instead for RTX 4090
    bf16=False,   # Better numerical stability than fp16

    load_best_model_at_end = True,       # MUST USE for early stopping
    metric_for_best_model = "eval_loss", # metric we want to early stop on
    greater_is_better = False,

    # Saving strategy
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    fp16_full_eval=True,        # Use bf16 for eval too
    bf16_full_eval=False,

    # Evaluation
    eval_strategy="steps",  # Disable to save memory
    per_device_eval_batch_size=2,
    eval_steps=20,

    # Logging
    logging_steps=25,
    logging_strategy="steps",
    report_to="tensorboard",  # Disable wandb/tensorboard for simplicity

    # Reproducibility
    seed=42,
    data_seed=42,
)

In [28]:
# Split into train/eval (90% train, 10% eval)
dataset_splits = dataset.train_test_split(
    test_size=0.2,  # 10% for evaluation
    seed=42,        # For reproducibility
    shuffle=True    # Shuffle before splitting
)

print(f"Train samples: {len(dataset_splits['train'])}")
print(f"Eval samples: {len(dataset_splits['test'])}")

Train samples: 9600
Eval samples: 2400


In [29]:
# ==============================================================================
# TRAINER SETUP
# ==============================================================================

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_splits['train'],
    eval_dataset=dataset_splits['test'],
    dataset_text_field="text",
    max_seq_length=512*2,
    dataset_num_proc=2,
    packing=False,  # Don't pack sequences for QA task
    args=training_args,
)


In [ ]:
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience = 3,     # How many steps we will wait if the eval loss doesn't decrease
                                     # For example the loss might increase, but decrease after 3 steps
    early_stopping_threshold = 0.0,  # Can set higher - sets how much loss should decrease by until
                                     # we consider early stopping. For eg 0.01 means if loss was
                                     # 0.02 then 0.01, we consider to early stop the run.
)
trainer.add_callback(early_stopping_callback)

In [30]:
# ==============================================================================
# TRAINING
# ==============================================================================

print("Starting training...")
trainer_stats = trainer.train()

# Print training statistics
print(f"Training completed!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,600 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Step,Training Loss,Validation Loss
20,No log,2.561419
40,2.843100,1.389258


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training completed!
Training time: 967.08 seconds
Samples per second: 0.83


In [ ]:
# ==============================================================================
# SAVE MODEL FOR VLLM/SGLANG COMPATIBILITY
# ==============================================================================

print("Saving model for vLLM/SGLang compatibility...")

# Save as merged 16-bit model (recommended for vLLM/SGLang)
model.save_pretrained_merged(
    "rbi-llama-3.2-3b-merged",
    tokenizer,
    save_method="merged_16bit",
)



In [ ]:
repo_id = "Vishva007/Llama-3.2-3B-Instruct-RBI-QA"

# Push to HuggingFace Hub (optional)
model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method="merged_16bit",
    token="token" # replace with your actual token
)

print("Model saved successfully!")


Saving model for vLLM/SGLang compatibility...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

In [ ]:

# ==============================================================================
# TEST INFERENCE
# ==============================================================================

print("Testing inference...")

# Enable inference mode
FastLanguageModel.for_inference(model)



In [ ]:
# Test with a sample RBI question
test_messages = [
    {"role": "system", "content": "You are an expert assistant specialized in Reserve Bank of India (RBI) regulations and banking policies."},
    {"role": "user", "content": "What are the key requirements for banks regarding capital adequacy ratio as per RBI guidelines?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )


In [ ]:

response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("Sample response:", response)

print("Fine-tuning complete! Model ready for vLLM/SGLang deployment.")